# Noisy diagnostics and classical simulability

In [ ]:
import itertools
from fractions import Fraction
from functools import reduce
from math import comb, log, log2, pi, sqrt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from scipy.linalg import schur
from scipy.sparse.linalg import LinearOperator, eigs

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.diag([1, -1]).astype(complex)
PAULI = [I2, X, Y, Z]
HAD = (X + Z) / np.sqrt(2)
SPH = np.diag([1, 1j]).astype(complex)
TG = np.diag([1, np.exp(1j * np.pi / 4)])

D_LOCAL = 2
C_TARGET = 3.0
TOL = 1e-3
ETA_NG = 1e-1
ALPHA = log2(4 / (2 + sqrt(2)))
M_WIT = 3
RNG = np.random.default_rng(20260907)

WIT_PS = np.concatenate([np.geomspace(0.0072, 0.10, 17),
                         np.array([0.12, 0.14, 0.16, 0.175, 0.185, 0.192,
                                   0.198, 0.202, 0.205, 0.25, 0.30])])
WIT_DEPTH, MOM_DEPTH, WIT_PLOT_DEPTH = 185, 25, 45
NG_LAMS, NG_DEPTH, NG_REAL = np.geomspace(0.002, 0.50, 25), 60, 20
NG8_LAMS, NG8_DEPTH, NG8_REAL = np.geomspace(0.006, 0.30, 10), 32, 5
MAP_RES = 240
ELL_NS = (6, 8, 10, 12, 14)
CROSS_NS = (6, 8, 10, 12, 14, 16, 18, 20, 22)
FUSION_NS = (4, 6, 8)
EXACT_NMAX = 12

def kron(ops):
    return reduce(np.kron, ops)

## Fourth-order Clifford commutant

Spanned by the 24 replica permutations and 6 Pauli monomials; it factorises over sites, so every overlap is the $N$-th power of a single-site trace.

In [ ]:
def perm_matrix(sigma):
    """t_sigma: replica permutation on four copies of one qubit (16 x 16)."""
    M = np.zeros((16, 16), dtype=complex)
    for b in range(16):
        bits = [(b >> (3 - i)) & 1 for i in range(4)]
        new = [bits[sigma[i]] for i in range(4)]
        M[sum(v << (3 - i) for i, v in enumerate(new)), b] = 1
    return M

def _coset_key(s):
    return frozenset(tuple(s[v[i]] for i in range(4)) for v in KLEIN)

S4 = list(itertools.permutations(range(4)))
KLEIN = [(0, 1, 2, 3), (1, 0, 3, 2), (2, 3, 0, 1), (3, 2, 1, 0)]
COSETS, _seen = [], set()
for _s in S4:
    if _coset_key(_s) not in _seen:
        _seen.add(_coset_key(_s))
        COSETS.append(_s)

def coset_index(s):
    k = _coset_key(s)
    for i, c in enumerate(COSETS):
        if _coset_key(c) == k:
            return i
    raise ValueError

def compose(a, b):
    return tuple(a[b[i]] for i in range(4))

def invert(a):
    r = [0] * 4
    for i in range(4):
        r[a[i]] = i
    return tuple(r)

def sign_of(sigma):
    p, s = 0, list(sigma)
    for i in range(4):
        for j in range(i + 1, 4):
            if s[i] > s[j]:
                p ^= 1
    return 1 - 2 * p

OMEGA = sum(kron([p] * 4) for p in PAULI)
BASIS = np.array([perm_matrix(s) for s in S4] + [OMEGA @ perm_matrix(s) / 2 for s in COSETS])
NB = len(BASIS)
PAIR = np.einsum('aji,bji->ab', BASIS.conj(), BASIS)
EPS_COSET = np.array([sign_of(s) for s in COSETS], dtype=float)
K4, K4D = kron([TG] * 4), kron([TG] * 4).conj().T

_PROJ_W = np.zeros((5, 256, 256), dtype=complex)
for _idx in itertools.product(range(4), repeat=4):
    _P = kron([PAULI[i] for i in _idx]).reshape(256)
    _PROJ_W[sum(1 for i in _idx if i > 0)] += np.outer(_P, _P.conj()) / 16

def depol4_super(r):
    return sum((r ** w) * _PROJ_W[w] for w in range(5))

def depol4(m, r):
    return (depol4_super(r) @ m.reshape(256)).reshape(16, 16)

def gram(n):
    return PAIR ** n

def commutant_report():
    print("SEC. IV A.  FOURTH-ORDER CLIFFORD COMMUTANT")
    print("basis size %d (24 permutations + 6 Pauli monomials Omega_4 T_kappa)" % NB)
    print("   N   rank(W)   cond(W)")
    for n in (1, 2, 3, 4, 5, 6):
        G = gram(n)
        print(f"  {n:2d}   {np.linalg.matrix_rank(G, tol=1e-8):7d}   {np.linalg.cond(G):.3e}")
    print("  W is invertible exactly for N >= 3; at N = 2 its rank is 29.")

commutant_report()

## The $T$-gate layer map

The layer map of the global model on the 30 commutant coefficients: block structure, spectrum, and an exact rational verification.

In [ ]:
_layer_cache = {}

def layer_maps(n, r, heisenberg=False):
    """(A_T, A_N, W^{-1}) for the global model on n qubits, every qubit noisy."""
    key = (n, round(float(r), 12), bool(heisenberg))
    if key in _layer_cache:
        return _layer_cache[key]
    Winv = np.linalg.inv(gram(n))
    k, kd = (K4D, K4) if heisenberg else (K4, K4D)
    gT = np.array([[np.trace(a.conj().T @ (k @ b @ kd)) for b in BASIS] for a in BASIS])
    sup = depol4_super(r)
    gN = np.array([[np.trace(a.conj().T @ (sup @ b.reshape(256)).reshape(16, 16))
                    for b in BASIS] for a in BASIS])
    res = (Winv @ (gT * PAIR ** (n - 1)), Winv @ (gN ** n), Winv)
    _layer_cache[key] = res
    return res

def lam_triple(D):
    """lambda_-, lambda_0, lambda_+ of the T-gate layer map."""
    return ((3 * D * D - 3 * D - 4) / (4 * (D * D - 1)),
            (3 * D * D - 4) / (4 * (D * D - 1)),
            (3 * D * D + 3 * D - 4) / (4 * (D * D - 1)))

def B_model(D):
    l0 = lam_triple(D)[1]
    return l0 * np.eye(6) - (D / (8 * (D * D - 1))) * (np.ones((6, 6))
                                                       - np.outer(EPS_COSET, EPS_COSET))

def right_mult_matrices():
    res = []
    for pi_ in S4:
        R = np.zeros((6, 6))
        for i, c in enumerate(COSETS):
            R[coset_index(compose(c, pi_)), i] = 1
        res.append(R)
    return res

def spectrum_report():
    print("SEC. IV C.  T-GATE LAYER MAP: BLOCK STRUCTURE, CENTRALITY, SPECTRUM")
    Rs = right_mult_matrices()
    print("   N   |A_perm-1|  |A[Om,perm]|  |B-model|  |[B,R_pi]|   lam_-        lam_+")
    for n in (3, 4, 5, 6):
        D = 2.0 ** n
        for heis in (False, True):
            A = np.real(layer_maps(n, 1.0, heisenberg=heis)[0])
            B = A[24:30, 24:30]
            comm = max(np.abs(B @ R - R @ B).max() for R in Rs)
            ev = np.sort(np.linalg.eigvalsh((B + B.T) / 2))
            full = np.sort(np.abs(np.linalg.eigvals(A)))
            n1 = int(np.sum(np.abs(full - 1) < 1e-9))
            print(f"  {n:2d}{'H' if heis else 'S'}  {np.abs(A[:24, :24]-np.eye(24)).max():.1e}"
                f"    {np.abs(A[24:30, :24]).max():.1e}     {np.abs(B-B_model(D)).max():.1e}"
                f"    {comm:.1e}   {ev[0]:.8f}  {ev[5]:.8f}   (#eig=1: {n1})")
        lm, l0, lp = lam_triple(D)
        print(f"      closed form lam_- = {lm:.8f}, lam_0 = {l0:.8f}, lam_+ = {lp:.8f}")
    print("\nWhich spectral mode each functional selects (Omega block of the input vector):")
    print("   N   product stabilizer: uniform / sign / rest   |  Heisenberg X_1: uniform / sign")
    for n in (3, 4, 5):
        Winv = np.linalg.inv(gram(n))
        o = kron([np.diag([1, 0]).astype(complex)] * 4)
        c0 = np.real(Winv @ (np.array([np.trace(b.conj().T @ o) for b in BASIS]) ** n))
        v = np.ones(NB, dtype=complex)
        for j in range(n):
            v = v * np.array([np.trace(b.conj().T @ kron([(X if j == 0 else I2)] * 4))
                              for b in BASIS])
        ch = np.real(Winv @ v)

        def dec(u):
            u = u[24:30]
            a = u @ np.ones(6) / 6
            b = u @ EPS_COSET / 6
            return a, b, np.abs(u - a - b * EPS_COSET).max()
        a1, b1, r1 = dec(c0)
        a2, b2, _ = dec(ch)
        print(f"  {n:2d}   {a1:+.3e} / {b1:+.1e} / {r1:.1e}      |  {a2:+.3e} / {b2:+.3e}")

def exact_rational_check(Nmax):
    """Exact rational verification of the T-gate layer map for D = 2^N (fractions)."""
    print(f"SEC. IV C.  EXACT RATIONAL VERIFICATION OF THE T-GATE MAP, N = 3..{Nmax}")
    a = np.rint(np.log2(np.real(PAIR))).astype(int)
    assert np.allclose(2.0 ** a, np.real(PAIR))
    gTf = np.real(np.array([[np.trace(x.conj().T @ (K4 @ y @ K4D)) for y in BASIS]
                            for x in BASIS]))
    gT = np.rint(gTf).astype(int)
    assert np.allclose(gT, gTf)
    print("  all single-site Gram entries are powers of two and all T-map entries integers")
    print("   N   perm block = 1   lower-left = 0   Omega block = closed form")
    for N in range(3, Nmax + 1):
        D = 2 ** N

        rows = [[Fraction(D ** int(a[i, j])) for j in range(30)]
                + [Fraction(int(gT[i, j]) * 2 ** (int(a[i, j]) * (N - 1))) for j in range(30)]
                for i in range(30)]
        for col in range(30):
            piv = next(r for r in range(col, 30) if rows[r][col] != 0)
            rows[col], rows[piv] = rows[piv], rows[col]
            inv = 1 / rows[col][col]
            rows[col] = [x * inv for x in rows[col]]
            for r in range(30):
                if r != col and rows[r][col] != 0:
                    f = rows[r][col]
                    rows[r] = [x - f * y for x, y in zip(rows[r], rows[col])]
        A = [row[30:] for row in rows]
        lam0 = Fraction(3 * D * D - 4, 4 * (D * D - 1))
        cc = Fraction(D, 4 * (D * D - 1))
        ok_om = all(A[24 + i][24 + j] == (lam0 if coset_index(compose(COSETS[i], invert(COSETS[j]))) == 0
                    else (-cc if EPS_COSET[coset_index(compose(COSETS[i], invert(COSETS[j])))] < 0
                          else 0))
                    for i in range(6) for j in range(6))
        ok_p = all(A[i][j] == (1 if i == j else 0) for i in range(24) for j in range(24))
        ok_l = all(A[i][j] == 0 for i in range(24, 30) for j in range(24))
        print(f"  {N:2d}      {str(ok_p):>5s}           {str(ok_l):>5s}            {str(ok_om):>5s}")

spectrum_report()
exact_rational_check(EXACT_NMAX)

## Replica damping and closed forms

The damping factor $\kappa_k$, the exponents $\alpha_k$, and closed forms for the IPR, purity and stabilizer purity, checked against the exact $30\times30$ map.

In [ ]:
def kappa(k, N, p, d=D_LOCAL):
    r = 1 - p
    D = float(d) ** N
    return ((1 + (d * d - 1) * r ** k) ** N - 1) / (D * D - 1)

def alpha_exp(k, p, d=D_LOCAL):
    return log(d * d / (1 + (d * d - 1) * (1 - p) ** k)) / log(d)

def A_closed(N, NT, p, d=D_LOCAL):
    D = float(d) ** N
    w = 1 + (d - 1) * (1 - p) ** 2
    return 1 + (w ** N - 1) / (D + 1) * kappa(2, N, p, d) ** (NT - 1)

def delta2_closed(N, NT, p, d=D_LOCAL):
    D = float(d) ** N
    return (1 + (D - 1) * kappa(2, N, p, d) ** NT) / D

def delta4_doped_noiseless(N, t):
    D = 2.0 ** N
    return (4 + (D - 1) * lam_triple(D)[0] ** t) / (D + 3)

def delta4_undoped_noisy(N, NT, p):
    D = 2.0 ** N
    return (1 + (D - 1) * kappa(4, N, p) ** NT) / D

def global_observables(n):
    z4 = np.array([np.trace((OMEGA / 2) @ b) for b in BASIS]) ** n
    z2 = np.array([np.trace(perm_matrix((1, 0, 2, 3)) @ b) for b in BASIS]) ** n
    return np.real(z4), np.real(z2)

def global_sweep(n, NT, p, nu):
    """Exact (Delta_2, Delta_4) after NT layers of the global model."""
    A_T, A_N, Winv = layer_maps(n, 1 - p)
    o = kron([np.diag([1, 0]).astype(complex)] * 4)
    c = Winv @ (np.array([np.trace(b.conj().T @ o) for b in BASIS]) ** n)
    z4, z2 = global_observables(n)
    for L in range(NT):
        for _ in range(nu):
            c = A_T @ c
        c = A_N @ c
    return float(np.real(z2 @ c)), float(np.real(z4 @ c))

def replica_report():
    print("SEC. III A.  REPLICA DAMPING IN THE GLOBAL MODEL")
    print("Exponent ratios alpha_{2m}/alpha_2 (-> m as p -> 0):")
    print("      p       a4/a2    a6/a2    a8/a2")
    for p in (0.001, 0.01, 0.05, 0.10, 0.20):
        a2 = alpha_exp(2, p)
        print(f"    {p:6.3f}   {alpha_exp(4, p)/a2:6.4f}  {alpha_exp(6, p)/a2:6.4f}  "
            f"{alpha_exp(8, p)/a2:6.4f}")
    print("\nClosed forms against the exact 30 x 30 map:")
    print("   N  N_T   p   nu |  Delta_2 exact    closed form     diff")
    for (n, NT, p, nu) in [(3, 1, 0.00, 0), (3, 3, 0.10, 0), (4, 3, 0.10, 0),
                           (4, 5, 0.05, 2), (5, 2, 0.20, 1), (5, 4, 0.02, 3)]:
        d2, _ = global_sweep(n, NT, p, nu)
        print(f"  {n:2d} {NT:3d} {p:5.2f} {nu:3d} |  {d2:.12f}  {delta2_closed(n, NT, p):.12f}"
            f"  {abs(d2-delta2_closed(n, NT, p)):.1e}")
    for n in (3, 4):
        A_T, _, Winv = layer_maps(n, 1.0)
        o = kron([np.diag([1, 0]).astype(complex)] * 4)
        c = Winv @ (np.array([np.trace(b.conj().T @ o) for b in BASIS]) ** n)
        z4, _ = global_observables(n)
        err = 0.0
        for t in range(1, 9):
            c = A_T @ c
            err = max(err, abs(np.real(z4 @ c) - delta4_doped_noiseless(n, t)))
        print(f"  Delta_4 doped noiseless, N={n}: max deviation over t=1..8: {err:.1e}")
    for n in (3, 4):
        for p in (0.05, 0.20):
            err = max(abs(global_sweep(n, NT, p, 0)[1] - delta4_undoped_noisy(n, NT, p))
                      for NT in (1, 2, 3, 4))
            print(f"  Delta_4 undoped noisy, N={n}, p={p:.2f}: max deviation N_T=1..4: {err:.1e}")
    print("\nDoped AND noisy: the fourth sector is NOT damped by kappa_4 alone.")
    print("  N   p    nu   F_4 after 3 layers (exact)   (D-1) kappa_4^3 (undoped formula)")
    for n in (4, 5):
        for nu in (0, 1, 2):
            p = 0.05
            D = 2.0 ** n
            _, d4 = global_sweep(n, 3, p, nu)
            print(f"  {n}  {p:.2f}  {nu}      {D*d4-1:.6e}                "
                f"{(D-1)*kappa(4, n, p)**3:.6e}")

replica_report()

## Label fusion

Exact fourth moments of a noisy doped brickwork. A two-qubit Clifford gate gives its two sites a common commutant label, so the chain carries $N/2+1$ blocks. Memory cost $30^{N/2+1}$, restricting this to $N\le8$.

In [ ]:
W2INV = np.linalg.pinv(PAIR ** 2, rcond=1e-10)
OBS4 = {"D4": OMEGA / 2,
        "D2": perm_matrix((1, 0, 2, 3)),
        "A": sum(kron([np.diag([1, 0]).astype(complex) if x == 0
                       else np.diag([0, 1]).astype(complex)] * 2 + [I2, I2])
                 for x in range(2))}

class Brickwork4:

    def __init__(self, N):
        assert N % 2 == 0, "even N"
        self.N = N
        self.S = [BASIS.copy() for _ in range(N)]
        self.blocks, self.c = None, None

    def apply_T(self, sites):
        for j in sites:
            self.S[j] = np.matmul(np.matmul(K4, self.S[j]), K4D)

    def apply_noise(self, sites, sup):
        for j in sites:
            self.S[j] = (self.S[j].reshape(NB, 256) @ sup.T).reshape(NB, 16, 16)

    def _gate_tensor(self, i, j):
        A = np.einsum('aji,sji->as', BASIS.conj(), self.S[i])
        B = np.einsum('aji,sji->as', BASIS.conj(), self.S[j])
        G = np.real(np.einsum('oa,as,at->ost', W2INV, A, B))
        self.S[i] = BASIS.copy()
        self.S[j] = BASIS.copy()
        return G

    def _first_layer(self, pairs, seed):
        legs, tens = [], []
        for (i, j) in pairs:
            v = np.array([np.trace(b.conj().T @ seed[i]) for b in BASIS])
            w = np.array([np.trace(b.conj().T @ seed[j]) for b in BASIS])
            tens.append(np.real(W2INV @ (v * w)))
            legs.append({i, j})
        c = tens[0]
        for t in tens[1:]:
            c = np.multiply.outer(c, t)
        self.blocks, self.c = legs, c

    def _apply_gate(self, i, j):
        bi = next(k for k, b in enumerate(self.blocks) if i in b)
        bj = next(k for k, b in enumerate(self.blocks) if j in b)
        G = self._gate_tensor(i, j)                          # G[o, s, t]
        c = np.moveaxis(self.c, [bi, bj], [0, 1])
        rest = c.shape[2:]
        c = np.ascontiguousarray(c).reshape(NB, NB, -1)       # c[s, t, R]
        li, lj = self.blocks[bi] - {i}, self.blocks[bj] - {j}
        if li and lj:      # out[s,t,o,R] = G[o,s,t] c[s,t,R]
            res = G.transpose(1, 2, 0)[:, :, :, None] * c[:, :, None, :]
            res = res.reshape((NB, NB, NB) + rest); nb = [li, lj]
        elif li:           # out[s,o,R] = sum_t G[o,s,t] c[s,t,R]
            res = np.matmul(G.transpose(1, 0, 2), c).reshape((NB, NB) + rest); nb = [li]
        elif lj:           # out[t,o,R] = sum_s G[o,s,t] c[s,t,R]
            res = np.matmul(G.transpose(2, 0, 1), c.transpose(1, 0, 2))
            res = res.reshape((NB, NB) + rest); nb = [lj]
        else:              # out[o,R] = sum_{s,t} G[o,s,t] c[s,t,R]
            res = (G.reshape(NB, NB * NB) @ c.reshape(NB * NB, -1)).reshape((NB,) + rest)
            nb = []
        keep = [b for k, b in enumerate(self.blocks) if k not in (bi, bj)]
        self.blocks, self.c = nb + [{i, j}] + keep, res

    def layer(self, parity, T_sites, noise_sites, sup, seed=None):
        pairs = [(i, i + 1) for i in range(parity, self.N - 1, 2)]
        if self.blocks is None:
            self._first_layer(pairs, seed)
        else:
            for (i, j) in pairs:
                self._apply_gate(i, j)
        if T_sites:
            self.apply_T(T_sites)
        if noise_sites:
            self.apply_noise(noise_sites, sup)

    def read(self, which):
        obs, res = OBS4[which], self.c
        for b in self.blocks:
            v = np.ones(NB)
            for j in b:
                v = v * np.real(np.einsum('ij,sji->s', obs, self.S[j]))
            res = np.tensordot(res, v, axes=([0], [0]))
        return float(res)

_bw_cache = {}

def brickwork_sweep(N, NT, p, nu, noisy="all"):
    key = (N, round(float(p), 12), nu, noisy)
    if key in _bw_cache and len(_bw_cache[key][0]) >= NT:
        A, D2, D4 = _bw_cache[key]
        return A[:NT], D2[:NT], D4[:NT]
    sup = depol4_super(1 - p)
    nsites = list(range(N)) if noisy == "all" else [0]
    seed = [kron([np.diag([1, 0]).astype(complex)] * 4) for _ in range(N)]
    bw = Brickwork4(N)
    A, D2, D4 = [], [], []
    for L in range(NT):
        bw.layer(L % 2, [(L * nu + k) % N for k in range(nu)], nsites if p > 0 else [],
                 sup, seed=seed if L == 0 else None)
        A.append(bw.read("A") * 2.0 ** N)
        D2.append(bw.read("D2"))
        D4.append(bw.read("D4"))
    res = (np.array(A), np.array(D2), np.array(D4))
    _bw_cache[key] = res
    return res

def witness(D2, D4, m=M_WIT):
    return m * np.log2(D2) - np.log2(D4)

def threshold_F2(D, m=M_WIT):
    """Value of D Delta_2 - 1 at the purity threshold Delta_2 = D^{-1/m}."""
    return D ** (1 - 1.0 / m) - 1

def witness_end(W):
    W = np.asarray(W)
    pos = np.nonzero(W > 0)[0]
    if not len(pos) or pos[-1] == len(W) - 1:
        return float("nan")
    k = pos[-1]
    return (k + 1) + W[k] / (W[k] - W[k + 1])

def first_crossing(y, level, below=True):
    y = np.asarray(y)
    idx = np.nonzero(y <= level)[0] if below else np.nonzero(y >= level)[0]
    if not len(idx):
        return float("nan")
    k = idx[0]
    if k == 0:
        return 1.0
    return k + (y[k - 1] - level) / (y[k - 1] - y[k])

def fusion_report():
    print("SEC. IV B.  LABEL FUSION: CONSISTENCY AND THE WITNESS")
    print("Undoped noiseless brickwork maps |0>^N to a stabilizer state: Delta_2 = Delta_4 = 1.")
    for (N, NT) in [(4, 2), (4, 4), (6, 4)]:
        _, d2, d4 = brickwork_sweep(N, NT, 0.0, 0)
        print(f"   N={N} N_T={NT}:  Delta_2 = {d2[-1]:.12f}   Delta_4 = {d4[-1]:.12f}")
    print("\nSoundness of W on the undoped noisy orbit (max must be <= 0):")
    worst = {2: -np.inf, 3: -np.inf}
    for n in (3, 4):
        for NT in (1, 2, 4, 8):
            for p in (0.0, 0.01, 0.05, 0.2, 0.5, 0.9):
                d2, d4 = global_sweep(n, NT, p, 0)
                for m in (2, 3):
                    worst[m] = max(worst[m], witness(d2, d4, m))
    for N in (4, 6):
        for p in (0.0, 0.05, 0.2):
            _, d2, d4 = brickwork_sweep(N, 12, p, 0, "all")
            worst[2] = max(worst[2], np.max(witness(d2, d4, 2)))
            worst[3] = max(worst[3], np.max(witness(d2, d4, 3)))
    print(f"   max W_3 (paper) = {worst[3]:+.2e}    max W_2 = {worst[2]:+.2e}")

fusion_report()

## Second moments and Pauli-path weights

Transfer matrices for the IPR and the purity, exact to $N=22$, the decay rate $\lambda_1$, and the generating function that gives the Pauli-path truncation order.

In [ ]:
def _gate2(d=D_LOCAL):
    """Two-site Clifford twirl on replica labels {e,s}: T(e|es) = T(s|es) = d/(d^2+1)."""
    q = d * d
    G = np.zeros((4, 4))
    for s1 in (0, 1):
        for s2 in (0, 1):
            j = 2 * s1 + s2
            if s1 == s2:
                G[3 * s1, j] = 1.0
            else:
                G[0, j] = d / (q + 1)
                G[3, j] = d / (q + 1)
    return G

def brick_second(N, NT, r, sites=None, d=D_LOCAL):
    D = float(d) ** N
    sites = list(range(N)) if sites is None else sites
    c = np.full([2] * N, (1.0 / (d * (d + 1))) ** N)
    G = _gate2(d)
    Nm = np.array([[1.0, (1 - r * r) / d], [0.0, r * r]])
    outA, outF = [], []
    for layer in range(NT):
        for (i, j) in [(i, i + 1) for i in range(layer % 2, N - 1, 2)]:
            c = np.moveaxis(c, [i, j], [0, 1]); sh = c.shape
            c = (G @ c.reshape(4, -1)).reshape(sh)
            c = np.moveaxis(c, [0, 1], [i, j])
        for j in sites:
            c = np.moveaxis(c, j, 0); sh = c.shape
            c = (Nm @ c.reshape(2, -1)).reshape(sh)
            c = np.moveaxis(c, 0, j)
        outA.append(D * D * float(np.sum(c)))
        t = c
        for _ in range(N):
            t = np.tensordot(t, np.array([d, d * d]), axes=([0], [0]))
        outF.append(D * float(t) - 1.0)
    return np.array(outA), np.array(outF)

def brick_A(N, NT, r, sites=None):
    return brick_second(N, NT, r, sites)[0][-1]

_GS = np.zeros((4, 4))
for _j, (_a, _b) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
    if _a == 0 and _b == 0:
        _GS[0, _j] = 1.0
    else:
        _GS[1, _j] = _GS[2, _j] = 0.2
        _GS[3, _j] = 0.6

def path_G(N, NT, r, sites, zs):
    """G(z) = sum_s z^{|s|} Wtilde_s, batched over the fugacity z.  G(1) = A."""
    zs = np.atleast_1d(np.asarray(zs, dtype=float))
    nz = len(zs)
    sites = set(range(N) if sites is None else sites)
    w = np.ones((nz,) + (2,) * N)
    fac = np.stack([np.ones(nz), zs], axis=1)
    for j in range(N):
        sh = [1] * (N + 1); sh[0] = nz; sh[j + 1] = 2
        w = w * fac.reshape(sh)
    for layer in range(NT):
        for (i, j) in [(i, i + 1) for i in range(layer % 2, N - 1, 2)]:
            w = np.moveaxis(w, [i + 1, j + 1], [1, 2]); sh = w.shape
            w = np.matmul(_GS, w.reshape(nz, 4, -1)).reshape(sh)
            w = np.moveaxis(w, [1, 2], [i + 1, j + 1])
        for j in range(N):
            f = zs * (r * r if j in sites else 1.0)
            sh = [1] * (N + 1); sh[0] = nz; sh[j + 1] = 2
            w = w * np.stack([np.ones(nz), f], axis=1).reshape(sh)
    res = w
    for _ in range(N):
        res = np.tensordot(res, np.array([1.0, 1 / 3.0]), axes=([1], [0]))
    return res

def _zgrid(r, npts=121, zmax=2.4):
    """The fugacity grid must contain z = 1/q explicitly (see App. D)."""
    zs = list(np.exp(np.linspace(0.0, zmax, npts)))
    q = r * r
    if q > 0 and 1.0 / q < np.exp(zmax):
        zs += [1.0 / q, 1.0 / q ** 0.5, min(1.0 / q ** 1.5, np.exp(zmax))]
    return np.unique(np.round(np.array(sorted(zs)), 12))

def ell_star_upper(N, NT, r, tol=TOL, sites=None):
    zs = _zgrid(r)
    gs = np.maximum(path_G(N, NT, r, sites, zs) - 1.0, 1e-300)
    lg, lz = np.log(gs), np.log(zs)
    bnd = lambda l: float(np.min(np.exp(lg - l * lz)))
    if bnd(0) <= tol:
        return 0
    lo, hi = 0, 8000
    while lo + 1 < hi:
        mid = (lo + hi) // 2
        if bnd(mid) <= tol:
            hi = mid
        else:
            lo = mid
    return hi

def ell_star_lower(N, NT, r, tol=TOL, sites=None, eps=1e-5):
    g = path_G(N, NT, r, sites, [1 - eps, 1.0, 1 + eps])
    T = float(g[1] - 1.0)
    m = (g[2] - g[0]) / (2 * eps) / T
    return m - tol * N * (NT + 1) / T

def decay_rate(N, r, sites, k=8):
    from scipy.sparse.linalg import LinearOperator, eigs
    G2, Nm = _gate2(), np.array([[1.0, (1 - r * r) / 2], [0.0, r * r]])

    def two_layer(v):
        c = v.reshape([2] * N)
        for parity in (0, 1):
            for (i, j) in [(i, i + 1) for i in range(parity, N - 1, 2)]:
                c = np.moveaxis(c, [i, j], [0, 1]); sh = c.shape
                c = (G2 @ c.reshape(4, -1)).reshape(sh)
                c = np.moveaxis(c, [0, 1], [i, j])
            for j in sites:
                c = np.moveaxis(c, j, 0); sh = c.shape
                c = (Nm @ c.reshape(2, -1)).reshape(sh)
                c = np.moveaxis(c, 0, j)
        return c.ravel()

    op = LinearOperator((2 ** N, 2 ** N), matvec=two_layer, dtype=float)
    ev = eigs(op, k=min(k, 2 ** N - 2), which='LM', return_eigenvectors=False,
              maxiter=20000, tol=1e-12)
    sub = [x for x in np.sort(np.abs(ev))[::-1] if x < 1 - 1e-8]
    return sqrt(sub[0]) if sub else float('nan')

def second_moment_report():
    print("SEC. V.  SECOND MOMENTS OF THE BRICKWORK")
    print("Doping-independence of the second moments (fourth-moment network, nu = 0 vs 2):")
    A0, D20, _ = brickwork_sweep(6, 8, 0.05, 0, "one")
    A2, D22, _ = brickwork_sweep(6, 8, 0.05, 2, "one")
    print(f"   max |A(nu=2)-A(nu=0)| = {np.abs(A2-A0).max():.1e},"
        f"  max |D2(nu=2)-D2(nu=0)| = {np.abs(D22-D20).max():.1e}")
    print("\nAsymptotic per-layer decay rate lambda_1 of A - 1, one noisy qubit:")
    print("      p      N = 10      N = 12    |  (1-p)^2")
    for p in (0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00):
        a = decay_rate(10, 1 - p, [0])
        b = decay_rate(12, 1 - p, [0])
        print(f"    {p:.2f}   {a:.6f}   {b:.6f}   |  {(1-p)**2:.4f}")
    print("Every qubit noisy:")
    for p in (0.02, 0.05, 0.10):
        print(f"    p={p:.2f}: N=10 {decay_rate(10, 1-p, list(range(10))):.6f}   "
            f"N=12 {decay_rate(12, 1-p, list(range(12))):.6f}   (1-p)^2 = {(1-p)**2:.4f}")
    print("\nPauli-path truncation order, N_T = 12, p = 0.05, eps^2 delta_c = 1e-3:")
    print("      N |  every qubit noisy  |  one qubit noisy")
    for N in ELL_NS:
        u_all = ell_star_upper(N, 12, 0.95, TOL, None)
        lo = ell_star_lower(N, 12, 0.95, TOL, [0])
        up = ell_star_upper(N, 12, 0.95, TOL, [0])
        print(f"     {N:3d} |       {u_all:4d}          |   {lo:6.1f} < ell* <= {up:4d}")
    print(f"  analytic uniform-noise value: {log(1/TOL)/log(1/0.95**2):.1f}")

second_moment_report()

## The purity threshold and the crossover

A positive witness requires $\mathrm{Tr}\,\rho^2>D^{-1/3}$. We locate the end of the window in which the witness is positive, compare it with that threshold, and obtain the crossover size $N^*$ under sparse and dense noise.

In [ ]:
def entropy_bound_depth(N, p, n_noisy):
    return N / (M_WIT * n_noisy * alpha_exp(2, p))

def threshold_report():
    print("SEC. III B.  THE PURITY THRESHOLD AND THE WITNESS WINDOW")
    print("Brickwork, N = 6, nu = 2, one noisy qubit, witness W = 3 log2 D2 - log2 D4.")
    print("Depth of the zero of W, depth at which Delta_2 reaches D^{-1/3}, the")
    print("entropy-production lower bound on that depth, C = D^2 F_4 / F_2^2 at the last depth")
    print("with W > 0, and F_2 there against the window (D^{2/3}-1, D^{2/3}+C] of Prop. 3.")
    print("   p    zero of W   threshold depth   bound N/(3 alpha_2)   C at last W>0   F2 there  window")
    D = 64.0
    CH = D * D * (4 * D / (D + 3) - 1) / (D - 1) ** 2
    for p in (0.02, 0.05, 0.10, 0.20):
        _, d2, d4 = brickwork_sweep(6, WIT_DEPTH, p, 2, "one")
        W = witness(d2, d4)
        z = witness_end(W)
        F2, F4 = D * d2 - 1, D * d4 - 1
        if not np.isfinite(z):
            print(f"  {p:.2f}   witness never positive"); continue
        k = int(np.floor(z))
        ceil = first_crossing(np.log(np.maximum(F2, 1e-300)), log(threshold_F2(D)))
        C = D * D * F4[k - 1] / F2[k - 1] ** 2
        print(f"  {p:.2f}   {z:8.2f}      {ceil:8.2f}          {entropy_bound_depth(6, p, 1):8.2f}"
            f"           {C:6.2f}      {F2[k-1]:7.2f}   ({D**(2/3)-1:.2f}, {D**(2/3)+C:.2f}]")
    print(f"  Haar value of C at N = 6: {CH:.3f}")
    print("\nFourth-to-second rate ratio ln r4/ln r2 of F_k = D Delta_k - 1 (N = 6, nu = 2):")
    print("   noisy set   p      r2          r4        ratio")
    for noisy in ("one", "all"):
        for p in (0.02, 0.05, 0.10):
            _, d2, d4 = brickwork_sweep(6, 44, p, 2, noisy)
            i, j = 29, 43
            F2, F4 = 64 * d2 - 1, 64 * d4 - 1
            if F4[j] < 1e-9:
                print(f"   {noisy:9s} {p:.2f}   F_4 below 1e-9 in the fitting window: not resolvable")
                continue
            r2 = (F2[j] / F2[i]) ** (1.0 / (j - i))
            r4 = (F4[j] / F4[i]) ** (1.0 / (j - i))
            print(f"   {noisy:9s} {p:.2f}   {r2:.6f}   {r4:.6f}   {log(r4)/log(r2):6.3f}")

_cross_cache = {}

def crossover_data(p=0.05, noisy="one"):
    key = (p, noisy)
    if key in _cross_cache:
        return _cross_cache[key]
    rows = []
    for N in CROSS_NS:
        sites = [0] if noisy == "one" else None
        depth = 110 if noisy == "one" else 40
        A, F2 = brick_second(N, depth, 1 - p, sites)
        D = 2.0 ** N
        lam = (F2[-1] / F2[-16]) ** (1 / 15.)
        u2 = first_crossing(np.log(A - 1), log(1e-2))
        u3 = first_crossing(np.log(A - 1), log(1e-3))
        cert = first_crossing(np.log(np.maximum(F2, 1e-300)), log(threshold_F2(D)))
        rows.append((N, lam, u2, u3, cert))
    fus = []
    for N in FUSION_NS:
        depth = int(12 + 4 * N) if noisy == "one" else 12
        _, d2, d4 = brickwork_sweep(N, depth, p, 2, noisy)
        W = witness(d2, d4)
        z = witness_end(W)
        D = 2.0 ** N
        if np.isfinite(z):
            k = int(np.floor(z))
            F2, F4 = D * d2[k - 1] - 1, D * d4[k - 1] - 1
            fus.append((N, z, D * D * F4 / F2 ** 2))
        else:
            fus.append((N, z, float("nan")))
    _cross_cache[key] = (rows, fus)
    return rows, fus

def linear_law(N, lam, m=M_WIT):
    """Threshold depth if F_2 decays at the asymptotic rate lam from D - 1."""
    D = 2.0 ** np.asarray(N, dtype=float)
    return np.log((D - 1) / (D ** (1 - 1.0 / m) - 1)) / log(1 / lam)

def crossing_size(rows, tol):
    lam = float(np.median([r[1] for r in rows][-3:]))
    u = float(np.median([r[2] if tol == 1e-2 else r[3] for r in rows[1:]]))
    Ns = np.arange(4, 80, 0.01)
    cert = linear_law(Ns, lam)
    idx = np.argmax(cert > u)
    return (Ns[idx] if cert[idx] > u else float("inf")), u, lam

def crossover_report():
    print("SEC. III C.  THE CROSSOVER")
    for noisy, label in (("one", "SPARSE NOISE (one noisy qubit)"),
                         ("all", "DENSE NOISE (every qubit noisy)")):
        rows, fus = crossover_data(0.05, noisy)
        print(f"\n{label}, brickwork, p = 0.05, nu = 2:")
        print("      N   rate lambda   N_T^unif(1e-2)  N_T^unif(1e-3)  N_T^thr (exact)  linear law")
        for (N, lam, u2, u3, cert) in rows:
            D = 2.0 ** N
            form = float(linear_law(N, lam))
            bnd = entropy_bound_depth(N, 0.05, 1 if noisy == "one" else N)
            print(f"     {N:2d}    {lam:.6f}       {u2:6.2f}          {u3:6.2f}          {cert:6.2f}"
                f"        {form:6.2f}     bound {bnd:6.2f}")
        if noisy == "one":
            for tol in (1e-2, 1e-3):
                Ns, u, lam = crossing_size(rows, tol)
                print(f"  crossing at eps^2 delta_c = {tol:g}:  N* = {Ns:.1f}"
                    f"   (leading term 3 log2(1/tol) = {3*log2(1/tol):.1f};"
                    f" entropy bound N* <= 3 alpha_2 N_T^unif = {3*alpha_exp(2, 0.05)*u:.1f})")
        print("  zero of W from label fusion (same circuit), threshold depth, and C = D^2 F4/F2^2 there:")
        for (N, z, C) in fus:
            c = [r[4] for r in rows if r[0] == N]
            if not c:
                A, F2 = brick_second(N, 110 if noisy == "one" else 30, 0.95,
                                     [0] if noisy == "one" else None)
                c = [first_crossing(np.log(np.maximum(F2, 1e-300)), log(threshold_F2(2.0 ** N)))]
            if np.isfinite(z):
                print(f"   N = {N}:  W changes sign at {z:6.2f};  threshold {c[0]:6.2f};  C = {C:.2f}")
            else:
                print(f"   N = {N}:  W never positive;  threshold {c[0]:6.2f}")
    print("\np-independence of N* under sparse noise (both depths are set by the same rate):")
    for pp in (0.02, 0.05, 0.10):
        lam = decay_rate(10, 1 - pp, [0])
        A, _ = brick_second(10, int(40 + 12 / (1 - lam)), 1 - pp, [0])
        u = first_crossing(np.log(A - 1), log(1e-3))
        Ns = np.arange(4, 80, 0.01)
        cert = linear_law(Ns, lam)
        print(f"   p = {pp:.2f}: lambda_1 = {lam:.4f}, N_T^unif(1e-3) = {u:6.2f}, "
            f"N* = {Ns[np.argmax(cert > u)]:.1f},  entropy bound N* <= {3*alpha_exp(2, pp)*u:.1f}")

threshold_report()
crossover_report()

## Matchgate circuits

Majorana sector weights, the SWAP degree walk, the two noise channels, and the non-Gaussianity of doped matchgate circuits by exact simulation.

In [ ]:
def A_matchgate(N, NT, lam):
    rt = 1 - 2 * lam
    return float(sum(comb(N, k) ** 2 / comb(2 * N, 2 * k) * rt ** (4 * k * NT)
                     for k in range(N + 1)))

_swap_cache = {}

def swap_degree_matrix(N):
    if N in _swap_cache:
        return _swap_cache[N]
    M = np.zeros((N + 1, N + 1))
    for k in range(N + 1):
        deg, tot = 2 * k, comb(2 * N, 2 * k)
        for m in range(5):
            if deg - m < 0 or deg - m > 2 * N - 4:
                continue
            pm = comb(4, m) * comb(2 * N - 4, deg - m) / tot
            mo = m if m in (0, 4) else 4 - m
            nd = deg - m + mo
            if 0 <= nd <= 2 * N and nd % 2 == 0:
                M[nd // 2, k] += pm
    _swap_cache[N] = M
    return M

def sector_damping(N, r):
    """D_2k(r) = E_{|S|=2k}[r^{|P(S)|}] for qubit depolarising noise, exact."""
    g = np.zeros(2 * N + 1)
    g[0] = 1.0
    for _ in range(N):
        new = np.zeros(2 * N + 1)
        for t in range(2 * N + 1):
            if g[t] == 0.0:
                continue
            odd = t % 2
            new[t] += g[t] * (r if odd else 1.0)
            if t + 1 <= 2 * N:
                new[t + 1] += g[t] * 2 * r
            if t + 2 <= 2 * N:
                new[t + 2] += g[t] * (1.0 if odd else r)
        g = new
    return np.array([g[2 * k] / comb(2 * N, 2 * k) for k in range(N + 1)])

def mg_evolve(N, NT, noise, nS, kind="deph"):
    Dm = (sector_damping(N, 1 - noise) if kind == "depol"
          else np.array([(1 - 2 * noise) ** (4 * k) for k in range(N + 1)]))
    Mp = np.linalg.matrix_power(swap_degree_matrix(N), nS) if nS else None
    W = np.array([float(comb(N, k)) for k in range(N + 1)])
    for _ in range(NT):
        if nS:
            W = Mp @ W
        W = Dm * W
    return W

def mg_A_kstar(N, NT, noise, nS, kind="deph", tol=TOL):
    W = mg_evolve(N, NT, noise, nS, kind)
    binv = np.array([comb(N, k) / comb(2 * N, 2 * k) for k in range(N + 1)])
    A = float(np.dot(binv, W))
    tail = np.cumsum((binv * W)[::-1])[::-1]
    ks = N
    for k in range(N + 1):
        if k + 1 >= N + 1 or tail[k + 1] <= tol:
            ks = k
            break
    return A, ks

def jw_joint(N):
    J = np.zeros((2 * N + 1, N + 1))
    J[0, 0] = 1.0
    for _ in range(N):
        new = np.zeros_like(J)
        for t in range(2 * N + 1):
            for w in range(N + 1):
                v = J[t, w]
                if v == 0:
                    continue
                odd = t % 2
                new[t, w + 1 if odd else w] += v
                if t + 1 <= 2 * N and w + 1 <= N:
                    new[t + 1, w + 1] += 2 * v
                if t + 2 <= 2 * N:
                    ww = w if odd else w + 1
                    if ww <= N:
                        new[t + 2, ww] += v
        J = new
    return J

def participation_ratio(N, NT, lam, nS):
    """Operator participation ratio R of W = Z_1 under the sector-uniform layer map."""
    W = np.zeros(N + 1); W[1] = 1.0
    Mp = np.linalg.matrix_power(swap_degree_matrix(N), nS) if nS else np.eye(N + 1)
    Dm = np.array([(1 - 2 * lam) ** (4 * k) for k in range(N + 1)])
    Rs = []
    for _ in range(NT):
        W = Dm * (Mp @ W)
        Rs.append(float(np.sum(W ** 2 / np.array([comb(2 * N, 2 * k) for k in range(N + 1)]))
                        / np.sum(W) ** 2))
    return np.array(Rs)

def matchgate_report():
    print("SEC. VI.  MATCHGATES: SECTOR WEIGHTS, SWAP WALK, NOISE CHANNELS")
    print("Collision-probability ratio of a pure matchgate circuit -> sqrt(pi N):")
    for N in (16, 64, 256, 1024):
        a = A_matchgate(N, 0, 0.0)
        print(f"   N = {N:5d}:  A = {a:9.4f}   sqrt(pi N) = {sqrt(pi*N):9.4f}"
            f"   relative deviation {abs(a/sqrt(pi*N)-1):.1e}")
    print("\nParity limit under repeated SWAPs, A_eq = 2 + (2^N-2)^2/(2^{2N-1}-2):")
    for N in (4, 8, 16):
        M = swap_degree_matrix(N)
        w = np.array([float(comb(N, k)) for k in range(N + 1)])
        for _ in range(3000):
            w = M @ w
        it = sum(comb(N, k) * w[k] / comb(2 * N, 2 * k) for k in range(N + 1))
        print(f"   N = {N:3d}:  iterated {it:.8f}   closed form "
            f"{2 + (2.0**N - 2)**2/(2.0**(2*N-1) - 2):.8f}")
    print("\nOperator participation ratio R of Z_1, N = 64, n_S = 1:")
    for lam in (0.0, 0.01, 0.05):
        R = participation_ratio(64, 50, lam, 1)
        print(f"   lambda = {lam:.2f}:  R after 1 layer = {R[0]:.2e},  min over 50 layers = {R.min():.2e}")
    print(f"   Haar value ~ 4^-N = {4.0**-64:.1e}")
    print("\nSector damping at degree 2, N = 64:  depolarising D_2 vs dephasing (1-2p)^4")
    for p in (0.01, 0.02, 0.05):
        print(f"   p = {p:.2f}:   {sector_damping(64, 1-p)[1]:.4f}    {(1-2*p)**4:.4f}")
    print("\nMean Jordan-Wigner weight at fixed Majorana degree:")
    print("      N    |S|=2    |S|=4    (N+2)/3")
    for N in (16, 32, 64):
        J = jw_joint(N)
        m = [float((J[t] * np.arange(N + 1)).sum() / J[t].sum()) for t in (2, 4)]
        print(f"    {N:4d}   {m[0]:6.2f}   {m[1]:6.2f}   {(N+2)/3:6.2f}")

def majoranas(n):
    g = []
    for j in range(n):
        Zs = [Z] * j
        g.append(kron(Zs + [X] + [I2] * (n - j - 1)))
        g.append(kron(Zs + [Y] + [I2] * (n - j - 1)))
    return g

def swap_matrix(n, site):
    D = 2 ** n
    S = np.zeros((D, D), dtype=complex)
    for b in range(D):
        bits = [(b >> (n - 1 - i)) & 1 for i in range(n)]
        bits[site], bits[site + 1] = bits[site + 1], bits[site]
        S[sum(v << (n - 1 - i) for i, v in enumerate(bits)), b] = 1
    return S

class MatchgateSim:
    """Exact density-matrix simulation of brickwork matchgates + SWAP + Majorana dephasing."""

    def __init__(self, n):
        self.n = n
        self.D = 2 ** n
        self.g = majoranas(n)
        self.swaps = [swap_matrix(n, s) for s in range(n - 1)]
        self.pairs = [(i, j) for i in range(2 * n) for j in range(i + 1, 2 * n)]
        self.GG = np.array([self.g[i] @ self.g[j] - self.g[j] @ self.g[i]
                            for (i, j) in self.pairs])

    @staticmethod
    def _haar_u2(rng):
        z = (rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))) / np.sqrt(2)
        q, r = np.linalg.qr(z)
        return q * (np.diag(r) / np.abs(np.diag(r)))

    def random_matchgate(self, bond, rng):
        """G(A,B): A on {|00>,|11>}, B on {|01>,|10>}, det A = det B."""
        A = self._haar_u2(rng)
        B = self._haar_u2(rng)
        B = B * np.sqrt(np.linalg.det(A) / np.linalg.det(B))
        G = np.zeros((4, 4), dtype=complex)
        G[0, 0], G[0, 3], G[3, 0], G[3, 3] = A[0, 0], A[0, 1], A[1, 0], A[1, 1]
        G[1, 1], G[1, 2], G[2, 1], G[2, 2] = B[0, 0], B[0, 1], B[1, 0], B[1, 1]
        return np.kron(np.kron(np.eye(2 ** bond), G), np.eye(2 ** (self.n - bond - 2)))

    def covariance(self, rho):
        n2 = 2 * self.n
        vals = (1j * np.einsum('ij,kji->k', rho, self.GG) / 2).real
        G = np.zeros((n2, n2))
        for v, (i, j) in zip(vals, self.pairs):
            G[i, j] = v
            G[j, i] = -v
        return G

    def nonGaussianity(self, rho):
        """delta = S(rho_G) - S(rho), in bits."""
        T, _ = schur(self.covariance(rho), output="real")
        nus = np.clip(np.abs(np.diag(T, 1)[::2]), 0.0, 1.0)
        q = (1 + nus) / 2
        with np.errstate(divide='ignore', invalid='ignore'):
            hb = np.nan_to_num(-(q * np.log2(q) + (1 - q) * np.log2(1 - q)))
        ev = np.linalg.eigvalsh(rho).real
        ev = ev[ev > 1e-14]
        return float(np.sum(hb)) + float(np.sum(ev * np.log2(ev)))

    def run(self, NT, lam, nS, rng, record=None):
        rho = np.zeros((self.D, self.D), dtype=complex)
        rho[0, 0] = 1.0
        res = {}
        record = set(range(1, NT + 1)) if record is None else set(record)
        for L in range(1, NT + 1):
            for bond in range((L - 1) % 2, self.n - 1, 2):
                U = self.random_matchgate(bond, rng)
                rho = U @ rho @ U.conj().T
            for s in range(nS):
                Sw = self.swaps[(L + s) % (self.n - 1)]
                rho = Sw @ rho @ Sw.conj().T
            if lam > 0:
                for gj in self.g:
                    rho = (1 - lam) * rho + lam * (gj @ rho @ gj)
            rho = rho / np.trace(rho).real
            if L in record:
                res[L] = self.nonGaussianity(rho)
        return res

_CACHE = {}

def ng_grid(n=6, nS=1, lams=None, NT=None, NR=None, tag=None):
    """delta(lambda, depth), averaged over circuit realisations."""
    tag = tag or f"ng{n}_{nS}"
    if tag in _CACHE:
        return _CACHE[tag]
    lams = NG_LAMS if lams is None else lams
    NT = NG_DEPTH if NT is None else NT
    NR = NG_REAL if NR is None else NR
    print(f"  computing non-Gaussianity at N = {n} ({len(lams)} x {NR}) ...", flush=True)
    sim = MatchgateSim(n)
    grid = np.zeros((len(lams), NT))
    for i, lm in enumerate(lams):
        acc = np.zeros(NT)
        for s in range(NR):
            o = sim.run(NT, float(lm), nS, np.random.default_rng(5000 * i + s))
            acc += np.array([o[d] for d in range(1, NT + 1)])
        grid[i] = acc / NR
    _CACHE[tag] = (np.asarray(lams), grid)
    return _CACHE[tag]

def ng_boundary(lams, grid, eta):
    lc, nc = [], []
    for i, lm in enumerate(lams):
        if grid[i, 0] <= eta:
            continue
        z = first_crossing(grid[i], eta)
        if np.isfinite(z):
            lc.append(lm)
            nc.append(z)
    return np.array(nc), np.array(lc)

def matchgate_labels(N, nS, lams, NTg):
    lab = np.zeros((len(lams), len(NTg)))
    uni = np.zeros_like(lab)
    Agrid = np.zeros_like(lab)
    for i, lm in enumerate(lams):
        for j, nt in enumerate(NTg):
            A, ks = mg_A_kstar(N, int(nt), float(lm), nS)
            Agrid[i, j] = A
            uni[i, j] = 1.0 if A - 1 <= TOL else 0.0
            if nS * nt <= C_TARGET * log2(N):
                lab[i, j] = 1
            elif 2 * ks <= C_TARGET:
                lab[i, j] = 2
            elif A - 1 <= TOL:
                lab[i, j] = 3
    return lab, uni, Agrid

def nongaussianity_report():
    print("SEC. VI.  FERMIONIC NON-GAUSSIANITY OF NOISY DOPED MATCHGATE CIRCUITS")
    sim = MatchgateSim(6)
    print("Soundness: no SWAP gates, delta must vanish (bits):")
    worst = 0.0
    for lam in (0.0, 0.02, 0.10):
        o = sim.run(8, lam, 0, np.random.default_rng(3), record=[1, 4, 8])
        worst = max(worst, max(abs(v) for v in o.values()))
        print(f"   lambda = {lam:.2f}:  " + "   ".join(f"{o[d]:+.1e}" for d in (1, 4, 8)))
    print(f"   max |delta| = {worst:.1e}")
    print("One SWAP per layer (mean of 4 realisations, bits):")
    for lam in (0.0, 0.02, 0.10):
        acc = {}
        for s in range(4):
            o = sim.run(8, lam, 1, np.random.default_rng(100 + s), record=[1, 4, 8])
            for k, v in o.items():
                acc.setdefault(k, []).append(v)
        print(f"   lambda = {lam:.2f}:  " + "   ".join(f"{np.mean(acc[d]):.4f}" for d in (1, 4, 8)))

    lams, grid = ng_grid()
    NT = grid.shape[1]
    lab, _, _ = matchgate_labels(6, 1, lams, np.arange(1, NT + 1))
    cov = lab > 0
    print("\nTwo failure directions at N = 6, n_S = 1, on the (lambda, depth) grid "
        f"({grid.size} cells):")
    print("   eta (bits)   uncovered & resource gone   covered & resource present")
    for eta in (0.01, 0.05, 0.10, 0.20):
        res = grid > eta
        print(f"     {eta:5.2f}              {int(np.sum(~cov & ~res)):5d}                  "
            f"{int(np.sum(cov & res)):5d}")

    print("\nSystem size: depth of the 0.1-bit boundary divided by the last uncovered depth,")
    print("per lambda (values > 1: the boundary lies beyond the uncovered region).")
    l8 = NG8_LAMS
    for n, tag, NTn, NR in ((6, "ng6_on8", NG8_DEPTH, NG8_REAL * 2),
                            (8, "ng8", NG8_DEPTH, NG8_REAL)):
        L, G = ng_grid(n, 1, l8, NTn, NR, tag=tag)
        labn, _, _ = matchgate_labels(n, 1, L, np.arange(1, NTn + 1))
        ratios = []
        for i in range(len(L)):
            unc = np.nonzero(labn[i] == 0)[0]
            z = first_crossing(G[i], 0.1) if G[i, 0] > 0.1 else float("nan")
            if len(unc) and np.isfinite(z):
                ratios.append(z / (unc[-1] + 1))
        rs = "  ".join(f"{x:.2f}" for x in ratios)
        med = np.median(ratios) if ratios else float("nan")
        print(f"   N = {n}:  median {med:.2f}   [{rs}]")

matchgate_report()
nongaussianity_report()

## Independent checks

Label fusion against explicit random Clifford circuits and against the second-moment and Pauli-path transfer matrices, the SWAP table against brute-force conjugation, and the matchgate sector formula against exact simulation.

In [ ]:
def _embed(op, n, q):
    return kron([op if j == q else I2 for j in range(n)])

def _cnot(n, c, t):
    D = 2 ** n
    M = np.zeros((D, D), dtype=complex)
    for b in range(D):
        bits = [(b >> (n - 1 - j)) & 1 for j in range(n)]
        if bits[c]:
            bits[t] ^= 1
        M[sum(v << (n - 1 - j) for j, v in enumerate(bits)), b] = 1
    return M

def brute_brickwork(N, NT, p, nu, nsamp=300, noisy="all"):
    D, r = 2 ** N, 1 - p
    PL = np.array([kron([PAULI[i] for i in l]) for l in itertools.product(range(4), repeat=N)])
    Hs = [_embed(HAD, N, q) for q in range(N)]
    Ss = [_embed(SPH, N, q) for q in range(N)]
    Ts = [_embed(TG, N, q) for q in range(N)]
    Pq = [[_embed(P, N, q) for P in PAULI] for q in range(N)]
    nsites = list(range(N)) if noisy == "all" else [0]
    A = S2 = S4 = 0.0
    for _ in range(nsamp):
        rho = np.zeros((D, D), dtype=complex)
        rho[0, 0] = 1
        for L in range(NT):
            for i in range(L % 2, N - 1, 2):
                U = np.eye(D, dtype=complex)
                for _g in range(80):
                    k = RNG.integers(0, 3)
                    if k == 0:
                        U = Hs[i + RNG.integers(0, 2)] @ U
                    elif k == 1:
                        U = Ss[i + RNG.integers(0, 2)] @ U
                    else:
                        a, b = (i, i + 1) if RNG.integers(0, 2) == 0 else (i + 1, i)
                        U = _cnot(N, a, b) @ U
                rho = U @ rho @ U.conj().T
            for k in range(nu):
                q = (L * nu + k) % N
                rho = Ts[q] @ rho @ Ts[q].conj().T
            for q in nsites:
                acc = np.zeros_like(rho)
                for P in Pq[q]:
                    acc += P @ rho @ P
                rho = r * rho + (1 - r) * acc / 4
        pd = np.real(np.diag(rho))
        A += D * np.sum(pd ** 2)
        a = np.real(np.einsum('ij,kji->k', rho, PL)) ** 2
        S2 += a.sum() / D
        S4 += (a ** 2).sum() / D
    return A / nsamp, S2 / nsamp, S4 / nsamp

def _brute_swap_check(n=4, site=1):
    D = 2 ** n
    g = majoranas(n)
    mons = {}
    for k in range(0, 2 * n + 1, 2):
        for S in itertools.combinations(range(2 * n), k):
            M = np.eye(D, dtype=complex)
            for j in S:
                M = M @ g[j]
            mons[S] = M
    SW = swap_matrix(n, site)
    T = np.zeros((n + 1, n + 1))
    cnt = np.zeros(n + 1)
    for K, MK in mons.items():
        Aop = SW.conj().T @ MK @ SW
        for K2, MK2 in mons.items():
            co = np.trace(MK2.conj().T @ Aop) / D
            if abs(co) > 1e-9:
                T[len(K2) // 2, len(K) // 2] += abs(co) ** 2
        cnt[len(K) // 2] += 1
    return float(np.max(np.abs(T / np.maximum(cnt, 1)[None, :] - swap_degree_matrix(n))))

def _gaussian_A_check(N=6, lam=0.05, NT=3, nreal=40):
    sim = MatchgateSim(N)
    acc = 0.0
    for s in range(nreal):
        rng = np.random.default_rng(900 + s)
        rho = np.zeros((sim.D, sim.D), dtype=complex); rho[0, 0] = 1
        for L in range(1, NT + 1):
            for bond in range((L - 1) % 2, N - 1, 2):
                U = sim.random_matchgate(bond, rng)
                rho = U @ rho @ U.conj().T
            for gj in sim.g:
                rho = (1 - lam) * rho + lam * (gj @ rho @ gj)
        acc += sim.D * np.sum(np.real(np.diag(rho)) ** 2)
    return acc / nreal

def validation_report():
    print("APPENDIX D.  INDEPENDENT CHECKS")
    print("(i) label fusion vs explicit Clifford circuits (N = 4, 300 realisations):")
    print("   N_T   p   nu |   A brute  A exact |  D2 brute D2 exact |  D4 brute D4 exact")
    for (NT, p, nu) in [(2, 0.00, 1), (3, 0.10, 1), (3, 0.10, 2)]:
        b = brute_brickwork(4, NT, p, nu, 300)
        A, d2, d4 = brickwork_sweep(4, NT, p, nu)
        print(f"   {NT:3d} {p:5.2f} {nu:3d} | {b[0]:9.5f} {A[-1]:8.5f} |"
            f" {b[1]:9.5f} {d2[-1]:8.5f} | {b[2]:9.5f} {d4[-1]:8.5f}")
    print("(ii) label fusion vs second-moment transfer matrix (collision probability):")
    err = 0.0
    for (N, p, nu) in [(6, 0.05, 0), (4, 0.10, 0), (6, 0.05, 2)]:
        A, _, _ = brickwork_sweep(N, 10, p, nu)
        for nt in (2, 4, 6, 10):
            err = max(err, abs(A[nt - 1] - brick_A(N, nt, 1 - p)))
    print(f"   max deviation {err:.1e}")
    print("(iii) replica vs Pauli-path transfer matrices:")
    err = 0.0
    for (N, NT, p) in [(4, 3, 0.1), (6, 4, 0.05), (8, 6, 0.02)]:
        err = max(err, abs(float(path_G(N, NT, 1 - p, None, [1.0])[0]) - brick_A(N, NT, 1 - p)))
    print(f"   max deviation {err:.1e}")
    print(f"(iv) SWAP degree matrix vs brute-force conjugation, N = 4: {_brute_swap_check():.1e}")
    print("(v) matchgate brickwork collision probability (N = 6, 40 realisations) against the")
    print("     sector formula, which assumes equilibration within each sector (Assumption 1):")
    for lam, NT in ((0.0, 1), (0.0, 3), (0.0, 8), (0.01, 6), (0.01, 12), (0.05, 3), (0.05, 6)):
        print(f"     lambda = {lam:.2f}, N_T = {NT:2d}:  simulation {_gaussian_A_check(6, lam, NT):.4f}"
            f"   formula {A_matchgate(6, NT, lam):.4f}")

validation_report()

## Figure 1: the simulability maps

In [ ]:
plt.rcParams.update({"font.size": 8.6, "axes.labelsize": 8.6, "axes.titlesize": 9.0,
                     "legend.fontsize": 6.4, "xtick.labelsize": 7.6, "ytick.labelsize": 7.6,
                     "figure.dpi": 130, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
                     "font.family": "serif", "hatch.linewidth": 0.5,
                     "mathtext.fontset": "dejavuserif", "axes.spines.top": False,
                     "axes.spines.right": False})

C_TN, C_SR, C_PP, C_UNI = "#cfe0f0", "#e5d5f0", "#d6ecd6", "#fbe3c4"
PAL = ["#1f4e79", "#b02418", "#1e7b3c", "#d17a00", "#6a3d9a"]

def _framed(leg):
    leg.get_frame().set_facecolor("white")
    leg.get_frame().set_alpha(0.93)
    leg.get_frame().set_edgecolor("0.6")
    leg.get_frame().set_linewidth(0.5)

def magic_boundary():
    ps, nts = [], []
    for p in WIT_PS:
        _, d2, d4 = brickwork_sweep(6, WIT_DEPTH, float(p), 2, "one")
        z = witness_end(witness(d2, d4))
        if np.isfinite(z):
            ps.append(float(p)); nts.append(z)
    _CACHE["magic"] = (np.array(nts), np.array(ps))
    return _CACHE["magic"]

def clifford_labels(N, NU, ps, NTg):
    lo_tn = C_TARGET * log2(N) / (3 * log2(D_LOCAL))
    lab = np.zeros((len(ps), len(NTg)))
    for i, p in enumerate(ps):
        A, _ = brick_second(N, int(NTg[-1]), 1 - p, [0])
        uni = (A - 1) <= TOL
        for j, nt in enumerate(NTg):
            if nt <= lo_tn:
                lab[i, j] = 1
            elif NU * nt <= C_TARGET * log2(N) / ALPHA:
                lab[i, j] = 2
            elif uni[j]:
                lab[i, j] = 3
    return lab

def fig_maps():
    """Figure 1."""
    fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.3))
    N, NU = 6, 2
    ps = np.geomspace(2e-3, 1.0, MAP_RES)
    NTg = np.arange(1, 51)
    lab = clifford_labels(N, NU, ps, NTg)
    ax[0].pcolormesh(NTg, ps, lab, shading="nearest", vmin=-0.5, vmax=3.5,
                     cmap=ListedColormap(["white", C_TN, C_SR, C_UNI]),
                     rasterized=True)
    ax[0].contourf(NTg, ps, (lab == 0).astype(float), levels=[.5, 1.5],
                   colors="none", hatches=["///"], alpha=0)
    ax[0].contour(NTg, ps, (lab == 3).astype(float), levels=[0.5], colors="0.35",
                  linewidths=1.1, linestyles=[(0, (1, 1.6))])
    mx, my = magic_boundary()
    ax[0].plot(mx, my, color="k", lw=2.0, ls=(0, (4, 2)), zorder=6)
    ax[0].set_yscale("log"); ax[0].set_xlim(0.5, 50.5); ax[0].set_ylim(2e-3, 1.0)
    ax[0].set_xlabel("circuit depth $N_T$"); ax[0].set_ylabel(r"depolarising strength $p$")
    ax[0].set_title("(a) doped Clifford brickwork: magic", fontsize=8.9)
    h1 = [Patch(fc=C_TN, ec="0.5", lw=.4, label="tensor networks"),
          Patch(fc=C_SR, ec="0.5", lw=.4, label="stabilizer rank"),
          Patch(fc=C_UNI, ec="0.5", lw=.4, label="uniform sampling"),
          Patch(fc="white", ec="0.5", lw=.4, hatch="///", label="none of these"),
          plt.Line2D([], [], color="0.35", lw=1.1, ls=(0, (1, 1.6)),
                     label="uniform-sampling boundary"),
          plt.Line2D([], [], color="k", lw=1.8, ls=(0, (4, 2)),
                     label="magic present to the left")]
    _framed(ax[0].legend(handles=h1, loc="lower left", bbox_to_anchor=(0.012, 0.02),
                         handlelength=1.4, borderpad=0.3, labelspacing=0.3))

    Nmg, nS = 6, 1
    lams = np.geomspace(2e-3, 0.5, MAP_RES)
    NTg2 = np.arange(1, 51)
    lab2, uni, _ = matchgate_labels(Nmg, nS, lams, NTg2)
    ax[1].pcolormesh(NTg2, lams, lab2, shading="nearest", vmin=-0.5, vmax=3.5,
                     cmap=ListedColormap(["white", C_SR, C_PP, C_UNI]),
                     rasterized=True)
    ax[1].contourf(NTg2, lams, (lab2 == 0).astype(float), levels=[.5, 1.5],
                   colors="none", hatches=["///"], alpha=0)
    ax[1].contour(NTg2, lams, uni, levels=[0.5], colors="0.35",
                  linewidths=1.1, linestyles=[(0, (1, 1.6))])
    gl, gg = ng_grid()
    bx, by = ng_boundary(gl, gg, ETA_NG)
    if len(bx):
        ax[1].plot(bx, by, color="k", lw=2.0, ls=(0, (4, 2)), zorder=6)
    ax[1].set_yscale("log"); ax[1].set_xlim(0.5, 50.5); ax[1].set_ylim(2e-3, 0.5)
    ax[1].set_xlabel("circuit depth $N_T$")
    ax[1].set_ylabel(r"Majorana dephasing strength $\lambda$")
    ax[1].set_title(r"(b) matchgate$+$SWAP brickwork: non-Gaussianity", fontsize=8.9)
    h2 = [Patch(fc=C_SR, ec="0.5", lw=.4, label="SWAP-count expansion"),
          Patch(fc=C_PP, ec="0.5", lw=.4, label="Majorana propagation"),
          Patch(fc="white", ec="0.5", lw=.4, hatch="///", label="none of these"),
          plt.Line2D([], [], color="0.35", lw=1.1, ls=(0, (1, 1.6)),
                     label="uniform-sampling boundary"),
          plt.Line2D([], [], color="k", lw=1.8, ls=(0, (4, 2)),
                     label="non-Gaussianity present to the left")]
    _framed(ax[1].legend(handles=h2, loc="lower left", bbox_to_anchor=(0.012, 0.02),
                         handlelength=1.8, borderpad=0.3, labelspacing=0.3))
    fig.tight_layout()
    plt.show()

fig_maps()

## Figure 2: second and fourth moments of the same circuit

In [ ]:
def fig_moments():
    """Figure 2."""
    fig, ax = plt.subplots(1, 3, figsize=(11.2, 3.0))
    N, NU = 6, 2
    dep = np.arange(1, MOM_DEPTH + 1)
    ps = [0.0, 0.02, 0.05, 0.10, 0.20]
    for i, p in enumerate(ps):
        A, _ = brick_second(N, MOM_DEPTH, 1 - p, [0])
        ax[0].semilogy(dep, np.maximum(A - 1, 1e-18), color=PAL[i], label=rf"$p={p:g}$")
    ax[0].axhline(TOL, color="0.4", lw=0.7, ls=":")
    ax[0].set_xlabel("depth $N_T$"); ax[0].set_ylabel(r"$\mathcal{A}-1$")
    ax[0].set_title("(a) second moment: IPR"); ax[0].set_ylim(1e-4, 20)
    _framed(ax[0].legend(loc="lower left"))
    for i, p in enumerate(ps):
        _, _, d4 = brickwork_sweep(N, MOM_DEPTH, p, NU, "one")
        ax[1].semilogy(dep, np.maximum(2.0 ** N * d4 - 1, 1e-18), color=PAL[i],
                       label=rf"$p={p:g}$")
    ax[1].set_xlabel("depth $N_T$"); ax[1].set_ylabel(r"$D\Delta_4-1$")
    ax[1].set_title("(b) fourth moment: stabilizer purity"); ax[1].set_ylim(1e-5, 100)
    _framed(ax[1].legend(loc="lower left"))
    Dd = WIT_PLOT_DEPTH
    for i, p in enumerate([0.0, 0.01, 0.02, 0.05, 0.10]):
        _, d2, d4 = brickwork_sweep(N, Dd, p, NU, "one")
        ax[2].plot(np.arange(1, Dd + 1), witness(d2, d4), color=PAL[i], label=rf"$p={p:g}$")
    ax[2].axhline(0.0, color="k", lw=0.9)
    ax[2].set_xlim(1, Dd); ax[2].set_ylim(-10, 5)
    ax[2].set_xlabel("depth $N_T$"); ax[2].set_ylabel(r"$\mathcal{W}$")
    ax[2].set_title("(c) magic witness")
    _framed(ax[2].legend(loc="lower left"))
    fig.tight_layout()
    plt.show()

fig_moments()